# Prediction Model
Predicting collaboration success (`log_streams`) as a function of:
- `distance` — hyperbolic distance between genres
- `log_pop_src` / `log_pop_tgt` — genre popularity in that market/year
- `log_weight` — collaboration frequency
- `market` — regional market
- `year` — year

Models compared:
1. Linear Regression (baseline)
2. Decision Tree
3. Random Forest
4. Gradient Boosting (XGBoost)


## 0. Imports & config

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings('ignore')

DATASET = 'model_dataset.csv'
RANDOM_STATE = 42
TEST_SIZE = 0.2

print('Imports OK')

## 1. Load & prepare data

In [ ]:
df = pd.read_csv(DATASET)
print(f'Loaded: {len(df)} rows')
print(f'Markets: {sorted(df["market"].unique())}')
print(f'Years:   {sorted(df["year"].unique())}')
print()

# encode market as integer (tree models handle this fine)
le_market = LabelEncoder()
df['market_enc'] = le_market.fit_transform(df['market'])

# features and target
FEATURES = ['distance', 'log_pop_src', 'log_pop_tgt', 'log_weight', 'market_enc', 'year']
TARGET   = 'log_streams'

X = df[FEATURES]
y = df[TARGET]

# train / test split — stratified by market so all markets appear in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE,
    stratify=df['market']
)

print(f'Train: {len(X_train)} rows | Test: {len(X_test)} rows')
print(f'Features: {FEATURES}')

## 2. Train & evaluate all models

In [ ]:
models = {
    'Linear Regression':   LinearRegression(),
    'Decision Tree':        DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE),
    'Random Forest':        RandomForestRegressor(n_estimators=200, max_depth=10,
                                                   random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting':    GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                                       learning_rate=0.05,
                                                       random_state=RANDOM_STATE),
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)

    # 5-fold cross-validation R² on training set
    cv_r2 = cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()

    results[name] = {'model': model, 'y_pred': y_pred,
                     'R²': r2, 'RMSE': rmse, 'MAE': mae, 'CV R²': cv_r2}
    print(f'{name:<25} R²={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}  CV_R²={cv_r2:.4f}')

print()
best = max(results, key=lambda k: results[k]['R²'])
print(f'Best model: {best} (R²={results[best]["R²"]:.4f})')

## 3. Model comparison plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
model_names = list(results.keys())
colors = ['#888888', '#2a9d8f', '#e63946', '#457b9d']

metrics = [('R²', 'higher is better'), ('RMSE', 'lower is better'), ('MAE', 'lower is better')]

for ax, (metric, note) in zip(axes, metrics):
    vals = [results[m][metric] for m in model_names]
    bars = ax.bar(model_names, vals, color=colors, edgecolor='white')
    ax.set_title(f'{metric} ({note})', fontsize=11)
    ax.tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(vals)*0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Model comparison on test set', fontsize=13)
plt.tight_layout()
# plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature importance
Which features drive predictions the most?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

tree_models = ['Decision Tree', 'Random Forest', 'Gradient Boosting']
feature_labels = ['distance', 'pop source', 'pop target', 'weight', 'market', 'year']
colors_fi = ['#e63946', '#f4a261', '#2a9d8f', '#457b9d', '#8338ec', '#e9c46a']

for ax, name in zip(axes, tree_models):
    model = results[name]['model']
    importances = model.feature_importances_
    sorted_idx = np.argsort(importances)[::-1]

    ax.bar([feature_labels[i] for i in sorted_idx],
           [importances[i] for i in sorted_idx],
           color=[colors_fi[i] for i in sorted_idx],
           edgecolor='white')
    ax.set_title(name, fontsize=11)
    ax.set_ylabel('Importance')
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Feature importance by model', fontsize=13)
plt.tight_layout()
# plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('=== Random Forest feature importance (ranked) ===')
rf_imp = results['Random Forest']['model'].feature_importances_
for feat, imp in sorted(zip(feature_labels, rf_imp), key=lambda x: -x[1]):
    print(f'  {feat:<15} {imp:.4f}  ({imp*100:.1f}%)')

## 5. Predicted vs actual (best model)

In [ ]:
best_model_name = max(results, key=lambda k: results[k]['R²'])
y_pred_best = results[best_model_name]['y_pred']

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, y_pred_best, alpha=0.15, s=8, color='#457b9d')

lims = [min(y_test.min(), y_pred_best.min()),
        max(y_test.max(), y_pred_best.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='perfect prediction')
ax.set_xlabel('Actual log10(avg streams)')
ax.set_ylabel('Predicted log10(avg streams)')
ax.set_title(f'{best_model_name} — predicted vs actual\nR²={results[best_model_name]["R²"]:.4f}')
ax.legend()
plt.tight_layout()
# plt.savefig('pred_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Distance importance — isolated view
The key thesis question: how much does distance alone contribute,
vs genre popularity?

In [ ]:
from sklearn.ensemble import RandomForestRegressor

feature_sets = {
    'distance only':          ['distance'],
    'popularity only':        ['log_pop_src', 'log_pop_tgt'],
    'distance + popularity':  ['distance', 'log_pop_src', 'log_pop_tgt'],
    'all features':           FEATURES,
}

ablation_results = []
for label, feats in feature_sets.items():
    m = RandomForestRegressor(n_estimators=200, max_depth=10,
                              random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(X_train[feats], y_train)
    y_pred = m.predict(X_test[feats])
    r2 = r2_score(y_test, y_pred)
    ablation_results.append({'Feature set': label, 'R²': round(r2, 4)})
    print(f'{label:<30} R²={r2:.4f}')

print()
print('Key insight: how much does distance add on top of popularity alone?')
r2_pop  = next(r['R²'] for r in ablation_results if r['Feature set'] == 'popularity only')
r2_both = next(r['R²'] for r in ablation_results if r['Feature set'] == 'distance + popularity')
print(f'  Popularity only:         R²={r2_pop:.4f}')
print(f'  Distance + popularity:   R²={r2_both:.4f}')
print(f'  Gain from adding distance: {r2_both - r2_pop:+.4f}')

# bar chart
fig, ax = plt.subplots(figsize=(9, 4))
labels = [r['Feature set'] for r in ablation_results]
r2s    = [r['R²'] for r in ablation_results]
bar_colors = ['#e63946', '#f4a261', '#2a9d8f', '#457b9d']
bars = ax.bar(labels, r2s, color=bar_colors, edgecolor='white')
for bar, val in zip(bars, r2s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('R²')
ax.set_title('Ablation study — contribution of each feature set (Random Forest)')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
# plt.savefig('ablation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Performance by market
Does the model perform differently across markets?

In [ ]:
best_model = results[best_model_name]['model']
test_df = X_test.copy()
test_df['y_true'] = y_test.values
test_df['y_pred'] = results[best_model_name]['y_pred']
test_df['market'] = df.loc[X_test.index, 'market'].values

market_results = []
for market, group in test_df.groupby('market'):
    r2   = r2_score(group['y_true'], group['y_pred'])
    rmse = np.sqrt(mean_squared_error(group['y_true'], group['y_pred']))
    market_results.append({'market': market, 'n': len(group),
                           'R²': round(r2, 4), 'RMSE': round(rmse, 4)})

mdf = pd.DataFrame(market_results).sort_values('R²', ascending=False)
print(mdf.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
bar_colors_m = ['#e63946', '#f4a261', '#2a9d8f', '#8338ec',
                '#457b9d', '#e9c46a', '#264653', '#a8dadc', '#6d6875']
bars = ax.bar(mdf['market'], mdf['R²'], color=bar_colors_m[:len(mdf)], edgecolor='white')
for bar, val in zip(bars, mdf['R²']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)
ax.axhline(results[best_model_name]['R²'], color='black',
           linewidth=1.5, linestyle='--')
ax.text(len(mdf)-0.5, results[best_model_name]['R²'] + 0.003,
        f'overall R²={results[best_model_name]["R²"]:.3f}',
        ha='right', va='bottom', fontsize=9)
ax.set_ylabel('R²')
ax.set_title(f'{best_model_name} — R² per market')
plt.tight_layout()
# plt.savefig('market_performance.png', dpi=150, bbox_inches='tight')
plt.show()